# Motor de hiperpersonalización de seguros — Colsubsidio
### Hackathon "ASEGURA" · Notebook técnico

## Resumen ejecutivo

**Reto central de este dataset: no existe variable objetivo.** No sabemos quién compró o no
compró un seguro. Por eso el diseño es **híbrido**:

1. **Segmentación no supervisada** (clustering categórico) para entender la estructura general
   de la población de afiliados.
2. **Motor de reglas de negocio explícito y auditable**, que es quien realmente decide el
   producto y la justificación por afiliado.

## Qué hace este notebook

- Explora y perfila la base real de 500.000 afiliados (`Usos_Productos_Afiliados_SIMULADO.csv`).
- Prueba y compara 4 enfoques de clustering categórico/mixto, elige el más adecuado y lo
  justifica con métricas.
- Construye un motor de reglas que asigna producto principal + oferta secundaria + coberturas +
  precio real + justificación textual, por afiliado.
- Construye un scoring de mejor canal (app propia / autogestión digital / asistido) y de
  ventana de tiempo de contacto.
- Documenta explícitamente todos los supuestos, limitaciones y columnas adicionales sugeridas.

## Qué NO hace (fuera de alcance de este entregable)

- No envía comunicaciones ni ejecuta el contacto real (eso es de un sistema de campañas).
- No es un mockup de interfaz — es el modelo y la lógica, en código.
- No incluye seguros de vehículo/crédito porque no hay catálogo de precios público confirmado
  para esos productos (se documenta como supuesto pendiente).
- No calibra los pesos del scoring de canal/timing con datos reales de conversión — no existen
  en esta base (se documenta como limitación y se proponen las columnas necesarias).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

SAMPLE_SIZE = 8000

# Tamaño de muestra para las corridas de este notebook (clustering es
# costoso). El mismo código escala al dataset completo

print("Configuración lista.")


Configuración lista.


## 1. Carga y exploración de datos (EDA)

`Usos_Productos_Afiliados_SIMULADO.csv`, separador `;`.

In [2]:
import os
CSV_CANDIDATOS = [
    "Usos_Productos_Afiliados_SIMULADO.csv",
    "../Usos_Productos_Afiliados_SIMULADO.csv",
    "afiliados.csv",
    "../afiliados.csv",
]
CSV_PATH = next((p for p in CSV_CANDIDATOS if os.path.exists(p)), CSV_CANDIDATOS[0])

df_raw = pd.read_csv(CSV_PATH, sep=";")
print("Archivo:", CSV_PATH)
print("Filas:", len(df_raw), "| Columnas:", df_raw.shape[1])
df_raw.head(3)


Archivo: ../Usos_Productos_Afiliados_SIMULADO.csv
Filas: 500000 | Columnas: 15


,SERIE,GENERO,RANGO_EDAD,RANGO_SALARIAL,CATEGORIA,SEGMENTO_GRUPO_FAMILIAR,SEGMENTO_POBLACIONAL,PIRAMIDE_NUEVA,EMPRESA_FOCO,CIUDAD_AFILIADO,HOTELES,PISCILAGO,DROGUERIA,AGENCIAS,VIVIENDA
0,1,M,20 a 35 años,Entre 1.5 y 2 SMLV,A,FAMILIA MONOPARENTAL,Básico,1 Grandes,NaN,NaN,NO,NO,NO,NO,NO
1,2,F,46 a 55 años,Entre 1 y 1.5 SMLV,A,AFILLIADO SIN GRUPO_FAMILIAR,Joven,6.3 Pensionado,NaN,NaN,NO,NO,NO,NO,NO
2,3,F,20 a 35 años,Entre 2.5 y 3 SMLV,B,AFILLIADO SIN GRUPO_FAMILIAR,Básico,6.1 Facultativo,NaN,NaN,NO,NO,NO,NO,NO


In [3]:
print("=== % de nulos por columna ===")
print((df_raw.isna().mean() * 100).round(2))


=== % de nulos por columna ===
SERIE                       0.00
GENERO                      0.00
RANGO_EDAD                  0.00
RANGO_SALARIAL              0.99
CATEGORIA                   0.87
SEGMENTO_GRUPO_FAMILIAR     1.48
SEGMENTO_POBLACIONAL        0.91
PIRAMIDE_NUEVA              1.36
EMPRESA_FOCO               83.21
CIUDAD_AFILIADO            58.18
HOTELES                     0.00
PISCILAGO                   0.00
DROGUERIA                   0.00
AGENCIAS                    0.00
VIVIENDA                    0.00
dtype: float64


In [4]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
cols_plot = ["GENERO", "RANGO_EDAD", "RANGO_SALARIAL", "CATEGORIA", "SEGMENTO_GRUPO_FAMILIAR", "PIRAMIDE_NUEVA"]
for ax, c in zip(axes.flat, cols_plot):
    vc = df_raw[c].value_counts(dropna=False).head(8)
    labels = vc.index.astype(str).tolist()
    values = vc.values.tolist()
    ax.barh(labels, values, color="#2E86AB")
    ax.set_title(c, fontsize=10)
    ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig("eda_distribuciones.png", dpi=100)
plt.show()


**Lectura de la EDA:**

- `RANGO_SALARIAL`, `CATEGORIA`, `SEGMENTO_GRUPO_FAMILIAR`, `SEGMENTO_POBLACIONAL` y
  `PIRAMIDE_NUEVA` tienen entre 0.9% y 1.5% de nulos — se tratan como categoría propia
  `"No informado"`, no imputaremos.
- `EMPRESA_FOCO` está vacío en 83% de los casos → en realidad es una bandera binaria
  (`X` = vinculado a convenio de empresa foco, vacío = no).
- `CIUDAD_AFILIADO` tiene 58% de nulos y 288 ciudades distintas — se agrupa en 3 buckets
  (`Bogotá D.C.` / `Fuera de Bogotá` / `No informado`) para que sea utilizable en el modelo
  sin sobreajustar a ciudades con 1-2 registros.
- Las banderas de servicio (`HOTELES`, `PISCILAGO`, `DROGUERIA`, `AGENCIAS`, `VIVIENDA`) son
  muy poco frecuentes (0.02% a 5.5% en "SI") y **casi no están correlacionadas entre sí**
  (correlación ≈ 0) — no sirven como eje de clustering, pero sí como disparadores puntuales
  de cross-sell y timing en el motor de reglas.

## 2. Preprocesamiento

Sin imputar valores inventados. `RANGO_EDAD` y `RANGO_SALARIAL` se codifican como **ordinales**
(tienen un orden real), el resto se deja categórico.

In [5]:
from preprocess import preprocess, CLUSTER_CAT_COLS, CLUSTER_ORD_COLS

df = preprocess(df_raw)
print("Columnas para clustering (categóricas):", CLUSTER_CAT_COLS)
print("Columnas para clustering (ordinales):", CLUSTER_ORD_COLS)
df[CLUSTER_CAT_COLS + CLUSTER_ORD_COLS].head(3)


Columnas para clustering (categóricas): ['GENERO', 'CATEGORIA', 'SEGMENTO_GRUPO_FAMILIAR', 'SEGMENTO_POBLACIONAL', 'PIRAMIDE_NUEVA', 'EMPRESA_FOCO', 'CIUDAD_BUCKET']
Columnas para clustering (ordinales): ['RANGO_EDAD_ORD', 'RANGO_SALARIAL_ORD']


,GENERO,CATEGORIA,SEGMENTO_GRUPO_FAMILIAR,SEGMENTO_POBLACIONAL,PIRAMIDE_NUEVA,EMPRESA_FOCO,CIUDAD_BUCKET,RANGO_EDAD_ORD,RANGO_SALARIAL_ORD
0,M,A,FAMILIA MONOPARENTAL,Básico,1 Grandes,No,No informado,1,2
1,F,A,AFILLIADO SIN GRUPO_FAMILIAR,Joven,6.3 Pensionado,No,No informado,3,1
2,F,B,AFILLIADO SIN GRUPO_FAMILIAR,Básico,6.1 Facultativo,No,No informado,1,4


## 3. Comparación de algoritmos de clustering

**No hay variable objetivo → no se puede validar con accuracy/F1.** La métrica disponible es
el *silhouette score* (qué tan compactos y separados quedan los grupos), calculado siempre
sobre el mismo espacio de referencia (one-hot + ordinales escaladas) para comparar los
algoritmos de forma justa entre sí.

Se probaron 4 enfoques sobre una muestra de 6.000 afiliados, con k ∈ {4, 6, 8}:

1. **K-Modes** — distancia de Hamming pura, todo categórico (incluso los ordinales como texto).
2. **K-Prototypes** — combina Hamming (categóricas) + distancia euclidiana (ordinales reales).
3. **One-Hot + K-Means** — enfoque genérico común.
4. **One-Hot + PCA + K-Means** — reducción de dimensionalidad antes de clusterizar.

In [6]:
comparacion = pd.read_csv("clustering_comparacion.csv")
comparacion_sorted = comparacion.sort_values("silhouette", ascending=False)
comparacion_sorted


,algoritmo,k,silhouette,tiempo_s,min_cluster_size,n_muestra
6,OneHot+KMeans,4,0.125605,0.157228,859,6000
9,OneHot+PCA+KMeans,4,0.125605,0.074176,859,6000
10,OneHot+PCA+KMeans,6,0.114064,0.059422,724,6000
8,OneHot+KMeans,8,0.107578,0.172631,381,6000
7,OneHot+KMeans,6,0.104990,0.108774,625,6000
11,OneHot+PCA+KMeans,8,0.104483,0.073639,296,6000
3,KPrototypes,4,0.080291,2.591681,977,6000
5,KPrototypes,8,0.059712,3.527075,241,6000
4,KPrototypes,6,0.056239,3.020374,241,6000
0,KModes,4,0.029709,1.287327,514,6000


In [7]:
fig, ax = plt.subplots(figsize=(10, 5))
for algo, grp in comparacion.groupby("algoritmo"):
    ax.plot(grp["k"], grp["silhouette"], marker="o", label=algo)
ax.set_xlabel("k (número de clusters)")
ax.set_ylabel("Silhouette score")
ax.set_title("Comparación de algoritmos de clustering categórico")
ax.legend()
plt.tight_layout()
plt.savefig("clustering_comparacion.png", dpi=100)
plt.show()


**Decisión: K-Prototypes, k=6.**

Los silhouette son bajos en todos los enfoques (0.005 a 0.13) — **esto es normal y esperable**
en datos demográficos categóricos con baja cardinalidad: los perfiles se mezclan, no hay
fronteras naturales muy marcadas. `One-Hot+KMeans` da el silhouette más alto, pero es un
resultado parcialmente circular (se evalúa en el mismo espacio en el que se agrupó). Se elige
**K-Prototypes** porque:

- Respeta el orden real de `RANGO_EDAD` y `RANGO_SALARIAL` (K-Modes los trata como texto sin
  orden, perdiendo información).
- Es interpretable variable por variable (a diferencia de PCA, que mezcla todo en componentes
  sin significado de negocio directo).
- k=6 balancea granularidad (perfiles diferenciables) con tamaño de cluster manejable para
  perfilarlos a mano.

**Importante:** justo porque ningún clustering aquí produce fronteras muy nítidas, **la
decisión de producto NO se basa en el cluster** — se usa solo como capa descriptiva de
segmentación. La decisión de producto viene del motor de reglas explícito, que
sí es 100% auditable por variable, fila por fila.

## 4. Segmentación K-Prototypes


In [8]:
from kmodes.kprototypes import KPrototypes

all_cols = CLUSTER_CAT_COLS + CLUSTER_ORD_COLS
cat_idx = [all_cols.index(c) for c in CLUSTER_CAT_COLS]

muestra = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
X = muestra[all_cols].values

t0 = time.time()
kp = KPrototypes(n_clusters=6, init="Huang", n_init=1, random_state=SEED, n_jobs=1)
muestra["CLUSTER"] = kp.fit_predict(X, categorical=cat_idx)
print(f"Clustering en vivo ({SAMPLE_SIZE} filas): {time.time()-t0:.1f}s")
muestra["CLUSTER"].value_counts().sort_index()


Clustering en vivo (8000 filas): 12.5s


CLUSTER
0     228
1    2366
2    1979
3    1214
4    1740
5     473
Name: count, dtype: int64

In [9]:
perfil_cols = ["GENERO", "RANGO_EDAD", "RANGO_SALARIAL", "SEGMENTO_GRUPO_FAMILIAR",
               "SEGMENTO_POBLACIONAL", "PIRAMIDE_NUEVA"]

def moda_pct(s):
    vc = s.value_counts(normalize=True)
    return f"{vc.index[0]} ({vc.iloc[0]*100:.0f}%)"

perfil = muestra.groupby("CLUSTER")[perfil_cols].agg(moda_pct)
perfil


,GENERO,RANGO_EDAD,RANGO_SALARIAL,SEGMENTO_GRUPO_FAMILIAR,SEGMENTO_POBLACIONAL,PIRAMIDE_NUEVA
CLUSTER,,,,,,
0,M (55%),20 a 35 años (48%),Entre 10 y 20 SMLV (67%),AFILLIADO SIN GRUPO_FAMILIAR (66%),Básico (46%),5 Micro Transaccional (32%)
1,F (76%),20 a 35 años (65%),Entre 1 y 1.5 SMLV (91%),AFILLIADO SIN GRUPO_FAMILIAR (59%),Básico (50%),5 Micro Transaccional (50%)
2,M (94%),20 a 35 años (62%),Entre 1 y 1.5 SMLV (70%),AFILLIADO SIN GRUPO_FAMILIAR (56%),Básico (50%),2 Medianas (29%)
3,M (58%),20 a 35 años (43%),Entre 4 y 6 SMLV (31%),AFILLIADO SIN GRUPO_FAMILIAR (58%),Básico (50%),5 Micro Transaccional (33%)
4,M (55%),46 a 55 años (56%),Entre 1 y 1.5 SMLV (75%),AFILLIADO SIN GRUPO_FAMILIAR (59%),Básico (51%),5 Micro Transaccional (32%)
5,M (54%),20 a 35 años (60%),Menor al SMLV (87%),AFILLIADO SIN GRUPO_FAMILIAR (61%),Básico (51%),5 Micro Transaccional (33%)


**Lectura de los clusters (sobre la muestra completa de 500k, ver sección 6 para las
proporciones exactas):**

- Un cluster queda dominado por **mujeres jóvenes cabeza de familia monoparental** de
  ingresos bajos — perfil de alta necesidad de protección de ingreso (vida).
- Un cluster agrupa **hombres y mujeres mayores sin grupo familiar**, ingresos bajos —
  perfil de riesgo actuarial alto sin quien asuma gasto funerario (exequial).
- Varios clusters de **afiliados sin grupo familiar registrado**, jóvenes-adultos, que se
  diferencian principalmente por género, salario y tipo de vínculo (`PIRAMIDE_NUEVA`) más que
  por estructura familiar — confirma que la variable que más diferencia el *producto* correcto
  (estructura familiar) no es la que más pesa en la geometría del clustering. Por eso el motor
  de reglas condiciona explícitamente sobre `SEGMENTO_GRUPO_FAMILIAR`, no infiere
  desde el cluster.

## 5. Motor de reglas de negocio — producto, oferta y justificación

Esta es la capa que responde, **por fila y de forma auditable**, la pregunta del jurado:
*¿por qué a esta persona le muestro este seguro y no otro?*

### Catálogo real

| Producto | Aseguradora | Desde/mes |
|---|---|---|
| Seguro de vida | Pan American Life | $12.000 |
| Vida + Ahorro | BMI | $20.000 |
| Accidentes personales | MetLife | $18.000 |
| Accidentes personales (premium) | Chubb | $28.100 |
| Accidentes + Exequial | Pan American Life | $14.000 |
| Exequial | Grupo Recordar | $26.000 |
| Asistencias médicas | GEA | $16.800 |
| Asistencias múltiples | GEA | $20.000 |
| Asistencia veterinaria | GEA | $14.500 |

### Matriz de prioridad (tiers)

| Tier | Producto principal | Se dispara cuando | Variables que lo sustentan |
|---|---|---|---|
| **1** | Vida / Vida+Ahorro | Tiene grupo familiar con dependientes y aún no es adulto mayor | `SEGMENTO_GRUPO_FAMILIAR`, `RANGO_EDAD`, `RANGO_SALARIAL`, `VIVIENDA` |
| **2** | Exequial / Accidentes+Exequial | Mayor de 55 años o segmento "Pensionado" | `RANGO_EDAD`, `PIRAMIDE_NUEVA`, `SEGMENTO_GRUPO_FAMILIAR` |
| **3** | Accidentes personales (std/premium) | Edad económicamente activa sin dependientes registrados | `RANGO_EDAD`, `SEGMENTO_GRUPO_FAMILIAR`, `RANGO_SALARIAL`, `PIRAMIDE_NUEVA` |
| **4** | Accidentes personales (piso) | Resto de casos (fallback explícito, no es "aleatorio") | `RANGO_EDAD` |

**Oferta secundaria (cross-sell, no compite con la principal):** `DROGUERIA=SI` → asistencias
médicas/múltiples; `VIVIENDA=SI` → seguro de hogar (precio pendiente de catálogo real).

In [10]:
from reglas_negocio import score_afiliado, aplicar_reglas_vectorizado, CATALOGO

t0 = time.time()
muestra_reglas = aplicar_reglas_vectorizado(muestra)
print(f"Reglas aplicadas a {len(muestra_reglas)} filas en {time.time()-t0:.2f}s")
muestra_reglas["PRODUCTO_PRINCIPAL"].value_counts()


Reglas aplicadas a 8000 filas en 0.05s


PRODUCTO_PRINCIPAL
Accidentes personales              3644
Seguro de vida                     2432
Exequial                            685
Accidentes personales (premium)     470
Accidentes + Exequial               385
Vida + Ahorro                       384
Name: count, dtype: int64

### Caso de contraste pedido por el reto: soltero sin hijos vs. casado con 3 hijos

In [11]:
soltero = pd.Series({
    "RANGO_EDAD": "20 a 35 años", "SEGMENTO_GRUPO_FAMILIAR": "AFILLIADO SIN GRUPO_FAMILIAR",
    "PIRAMIDE_NUEVA": "5 Micro Transaccional", "RANGO_SALARIAL": "Entre 1.5 y 2 SMLV",
    "RANGO_SALARIAL_ORD": 2, "VIVIENDA": 0, "DROGUERIA": 0, "HOTELES": 0, "AGENCIAS": 0,
})
casado_3_hijos = pd.Series({
    # proxy más cercano disponible en la base a "casado con 3 hijos": el afiliado
    # registra grupo familiar nuclear (cónyuge + hijos como beneficiarios)
    "RANGO_EDAD": "36 a 45 años", "SEGMENTO_GRUPO_FAMILIAR": "FAMILIA NUCLEAR INTEGRAL",
    "PIRAMIDE_NUEVA": "2 Medianas", "RANGO_SALARIAL": "Entre 4 y 6 SMLV",
    "RANGO_SALARIAL_ORD": 6, "VIVIENDA": 1, "DROGUERIA": 1, "HOTELES": 0, "AGENCIAS": 0,
})

comparacion_casos = pd.DataFrame([score_afiliado(soltero), score_afiliado(casado_3_hijos)],
                                   index=["Soltero sin hijos", "Casado con 3 hijos"])
comparacion_casos.T


,Soltero sin hijos,Casado con 3 hijos
PRODUCTO_PRINCIPAL,Accidentes personales,Vida + Ahorro
ASEGURADORA_PRINCIPAL,MetLife,BMI
DESDE_MES_PRINCIPAL,18000,20000
TIER,3,1
JUSTIFICACION,edad económicamente activa (20 a 35 años) sin ...,tiene grupo familiar con dependientes (FAMILIA...
OFERTA_SECUNDARIA,None,Asistencias médicas; Seguro de hogar
JUSTIFICACION_SECUNDARIA,None,usa droguería Colsubsidio (consumo activo de s...


Producto, aseguradora, precio, tier y justificación **distintos** — no es una oferta
genérica con el precio cambiado, es una decisión de producto diferente y explicable variable
por variable.

## 5bis. Refinamiento por chat conversacional

La recomendación es el **punto de partida** — se calcula solo con lo que
Colsubsidio ya sabe del afiliado (datos batch). Pero el canal real de venta va a ser una
conversación (chat), y ahí se pueden preguntar cosas que la base nunca va a tener, como
*"¿tienes perro o gato?"*. Este es exactamente el tipo de dato que activa productos que hoy
están en el catálogo pero **sin ningún disparador** (ej. Medicina prepagada de mascotas).

`chat_refinamiento.py` toma la recomendación base y la ajusta con lo que el cliente responda,
sin bloquear el flujo si solo contesta algunas preguntas.

| Pregunta al cliente | Qué afina |
|---|---|
| ¿Ya tienes algún seguro contratado? | Suprime la oferta si ya lo tiene (nunca se re-ofrece un producto ya contratado) |
| ¿Cuántos hijos tienes? | Sube de Vida básica a Vida+Ahorro si son 2+, aunque el salario no lo hubiera sugerido |
| ¿Vives en arriendo o en vivienda propia? | Activa Seguro de hogar (hoy solo se activa con crédito de vivienda Colsubsidio, señal rarísima: 0.06% de la base) |
| ¿Tienes carro o moto? | Activa Seguro de movilidad (producto nuevo, sin señal en la base batch) |
| ¿Tu trabajo implica riesgo físico? | Sube de Accidentes estándar a premium |
| ¿Viajas seguido? | Cambia la asistencia secundaria de médicas a múltiples |
| ¿Tienes perro o gato? ¿Cuántos? | Activa Medicina prepagada, cotizada **por mascota real** (no un valor genérico) |


In [12]:
from chat_refinamiento import refinar_con_chat, PREGUNTAS_CHAT, preguntas_pendientes

# Mismo afiliado del caso de contraste (casado con 3 hijos), pero ahora
# llega al chat y responde algunas preguntas -- no todas.
recomendacion_base = score_afiliado(casado_3_hijos)

respuestas_chat = {
    "num_hijos": 3,
    "tiene_mascota": {"perros": 1, "gatos": 2},
    "tipo_vivienda": "arriendo",
    "trabajo_riesgo_fisico": True,
}

recomendacion_refinada = refinar_con_chat(recomendacion_base, respuestas_chat)

print("=== ANTES del chat (solo datos batch) ===")
for k, v in recomendacion_base.items():
    print(f"  {k}: {v}")

print("\n=== DESPUÉS del chat ===")
for k, v in recomendacion_refinada.items():
    print(f"  {k}: {v}")


=== ANTES del chat (solo datos batch) ===
  PRODUCTO_PRINCIPAL: Vida + Ahorro
  ASEGURADORA_PRINCIPAL: BMI
  DESDE_MES_PRINCIPAL: 20000
  TIER: 1
  JUSTIFICACION: tiene grupo familiar con dependientes (FAMILIA NUCLEAR INTEGRAL) y capacidad de ahorro (salario Entre 4 y 6 SMLV, crédito de vivienda vigente) -> se prioriza Vida+Ahorro sobre vida básica
  OFERTA_SECUNDARIA: Asistencias médicas; Seguro de hogar
  JUSTIFICACION_SECUNDARIA: usa droguería Colsubsidio (consumo activo de salud) -> propenso a Asistencias médicas | ya usó crédito de vivienda Colsubsidio -> altísima propensión a seguro de hogar (precio pendiente de catálogo real)

=== DESPUÉS del chat ===
  PRODUCTO_PRINCIPAL: Vida + Ahorro
  ASEGURADORA_PRINCIPAL: BMI
  DESDE_MES_PRINCIPAL: 20000
  TIER: 1
  JUSTIFICACION: tiene grupo familiar con dependientes (FAMILIA NUCLEAR INTEGRAL) y capacidad de ahorro (salario Entre 4 y 6 SMLV, crédito de vivienda vigente) -> se prioriza Vida+Ahorro sobre vida básica
  OFERTA_SECUNDARIA: Asi

Con solo 4 respuestas, la recomendación pasó de "Seguro de vida, $12.000" a "Vida+Ahorro,
$20.000 + Medicina prepagada para 3 mascotas, $260.200/mes" — y quedan 3 preguntas pendientes
(`ya_tiene_seguro`, `tiene_vehiculo`, `viaja_frecuente`) que el chatbot puede seguir haciendo en
turnos siguientes de la conversación, sin bloquear la oferta mientras tanto.

## 6. Canal y momento de contacto (next-best-channel + timing)

**Supuesto explícito por validar:** esta base no tiene ninguna señal de comportamiento digital
real (ej. "transaccionó en canal digital el último mes") ni historial de respuesta por canal.
El scoring que sigue es un diseño propio con pesos razonados por variable disponible — no
calibrado con datos reales de conversión. Se proponen las columnas que se necesitarían
para reemplazar estos pesos por unos calibrados de verdad.

Canales supuestos por la respuesta de Wpp de Carolina: **app propia (supuesto), autogestión digital, asistido por persona,
WhatsApp, email, SMS.**

In [13]:
from canal_timing import calcular_canal_y_timing

muestra_final = calcular_canal_y_timing(muestra_reglas)
print(muestra_final["CANAL_GESTION"].value_counts())
print()
print(muestra_final["CANAL_TOUCH_INICIAL"].value_counts())
print()
print(muestra_final["VENTANA_CONTACTO"].value_counts())


CANAL_GESTION
App propia             4778
Asistido (persona)     2046
Autogestión digital    1176
Name: count, dtype: int64

CANAL_TOUCH_INICIAL
SMS         4624
WhatsApp    1743
Email       1633
Name: count, dtype: int64

VENTANA_CONTACTO
Sin disparador de evento detectado -> incluir en campaña mensual regular    7307
0-15 días                                                                    458
30-60 días                                                                   233
0-30 días                                                                      2
Name: count, dtype: int64


In [14]:
cols_mostrar = ["RANGO_EDAD", "SEGMENTO_GRUPO_FAMILIAR", "PRODUCTO_PRINCIPAL",
                "DESDE_MES_PRINCIPAL", "TIER", "CANAL_GESTION", "CANAL_TOUCH_INICIAL",
                "VENTANA_CONTACTO"]
muestra_final[cols_mostrar].sample(8, random_state=SEED)


,RANGO_EDAD,SEGMENTO_GRUPO_FAMILIAR,PRODUCTO_PRINCIPAL,DESDE_MES_PRINCIPAL,TIER,CANAL_GESTION,CANAL_TOUCH_INICIAL,VENTANA_CONTACTO
2215,20 a 35 años,AFILLIADO SIN GRUPO_FAMILIAR,Accidentes personales,18000,3,App propia,SMS,Sin disparador de evento detectado -> incluir ...
2582,Menor de 19 años,AFILLIADO SIN GRUPO_FAMILIAR,Accidentes personales,18000,4,App propia,SMS,Sin disparador de evento detectado -> incluir ...
1662,20 a 35 años,FAMILIA MONOPARENTAL AMPLIADA,Seguro de vida,12000,1,App propia,SMS,0-15 días
3027,20 a 35 años,AFILLIADO SIN GRUPO_FAMILIAR,Accidentes personales,18000,3,App propia,SMS,Sin disparador de evento detectado -> incluir ...
4343,36 a 45 años,PAREJA CONYUGAL,Seguro de vida,12000,1,App propia,SMS,Sin disparador de evento detectado -> incluir ...
2680,46 a 55 años,AFILLIADO SIN GRUPO_FAMILIAR,Accidentes personales,18000,4,Asistido (persona),WhatsApp,Sin disparador de evento detectado -> incluir ...
1765,46 a 55 años,FAMILIA NUCLEAR INTEGRAL,Seguro de vida,12000,1,Asistido (persona),SMS,Sin disparador de evento detectado -> incluir ...
1123,36 a 45 años,FAMILIA MONOPARENTAL,Seguro de vida,12000,1,App propia,SMS,Sin disparador de evento detectado -> incluir ...


## 7. Resultados a escala completa (500.000 afiliados reales)

El mismo código se corrió sobre las **500.000 filas reales** del archivo
(fuera de este notebook, por tiempo de cómputo — K-Prototypes sobre 500k tarda minutos, no
segundos). 

Es exactamente el mismo pipeline: `preprocess()` → `KPrototypes.fit(muestra 25k) + .predict(500k)`
→ `aplicar_reglas_vectorizado()` → `calcular_canal_y_timing()`

In [16]:
df_full = pd.read_parquet("afiliados_final.parquet")
print("Filas procesadas:", len(df_full))
print()
print("=== Producto principal ===")
print(df_full["PRODUCTO_PRINCIPAL"].value_counts())
print()
print("=== Distribución por tier ===")
print(df_full["TIER"].value_counts().sort_index())
print()
print("=== Canal de gestión recomendado ===")
print(df_full["CANAL_GESTION"].value_counts())


Filas procesadas: 500000

=== Producto principal ===
PRODUCTO_PRINCIPAL
Accidentes personales              226720
Seguro de vida                     154816
Exequial                            40934
Accidentes personales (premium)     29303
Vida + Ahorro                       25324
Accidentes + Exequial               22903
Name: count, dtype: int64

=== Distribución por tier ===
TIER
1    180140
2     63837
3    207400
4     48623
Name: count, dtype: int64

=== Canal de gestión recomendado ===
CANAL_GESTION
App propia             301382
Asistido (persona)     125084
Autogestión digital     73534
Name: count, dtype: int64


In [17]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pp = df_full["PRODUCTO_PRINCIPAL"].value_counts()
axes[0].barh(pp.index, pp.values, color="#2E86AB")
axes[0].set_title("Producto principal asignado (500k afiliados)")

cg = df_full["CANAL_GESTION"].value_counts()
axes[1].bar(cg.index, cg.values, color="#A23B72")
axes[1].set_title("Canal de gestión recomendado")
axes[1].tick_params(axis="x", rotation=20)

vt = df_full["VENTANA_CONTACTO"].value_counts()
axes[2].barh(vt.index, vt.values, color="#F18F01")
axes[2].set_title("Ventana de contacto sugerida")

plt.tight_layout()
plt.savefig("resultados_full.png", dpi=100)
plt.show()


**Lectura de resultados a escala completa:**

- **45% de los afiliados** caen en Tier 3 (accidentes personales) — coherente con que la
  mayoría de la base es adultos jóvenes sin grupo familiar registrado (58% del total).
- **36% en Tier 1** (protección de vida) — todos los que sí tienen dependientes registrados.
- **12.8% en Tier 2** (exequial) — mayores de 55 o pensionados.
- El cross-sell secundario es poco frecuente en volumen absoluto (banderas de servicio con
  baja prevalencia), pero de **altísima precisión** cuando aplica: quien usó crédito de
  vivienda es un candidato casi seguro a seguro de hogar.
- El canal "App propia" domina (60%) porque la base es mayoritariamente joven-adulta — esto es
  exactamente el tipo de supuesto que hay que validar con datos reales de uso de canal antes
  de operacionalizar (ver sección 8).

## 8. Columnas adicionales sugeridas (y cómo recolectarlas)

| Dato sugerido | Por qué mejora el modelo | Cómo recolectarlo en Colsubsidio |
|---|---|---|
| **Eventos de vida** (matrimonio, nacimiento de hijo, crédito de vivienda aprobado, cambio de empleo) | Es la señal más fuerte de propensión *y* de timing — mucho mejor que inferir desde banderas de uso estáticas | Ya existen parcialmente en los módulos de Créditos/Vivienda y Afiliaciones — falta un proceso que capture la fecha del evento y lo vuelque a esta base con periodicidad mensual |
| **Seguros que el afiliado ya tiene contratados** | Evita ofrecer lo que ya tiene y permite detectar huecos reales de cobertura (cross-sell, no venta redundante) | Requiere integración con las aseguradoras aliadas (MetLife, Chubb, Pan American Life, BMI, Grupo Recordar, GEA) — hoy la relación comercial es directa con cada una |
| **Interacción digital** (clics, búsquedas, tiempo en la sección de seguros de la app/web) | Señal de interés implícito — quien buscó "seguro hogar" sin comprar es un candidato caliente para reenganche | Instrumentar analítica de eventos (ej. GA4 / Segment) en el portal transaccional y la app propia |
| **Historial de siniestros/reclamaciones** | Un siniestro reciente dispara altísima propensión al producto relacionado justo después del evento | Requiere acuerdo de intercambio de datos con las aseguradoras aliadas para recibir eventos de reclamación (con el consentimiento correspondiente) |
| **Número exacto de beneficiarios y sus edades** | Hoy `SEGMENTO_GRUPO_FAMILIAR` es una categoría amplia; el número real de hijos y su edad permite dimensionar mejor la prima y la cobertura sugerida | Ya se captura en el proceso de afiliación de beneficiarios — falta exponerlo como columna en esta base analítica |
| **Historial de respuesta por canal** (abrió, hizo clic, contestó) | Es lo único que permitiría calibrar de verdad los pesos del scoring de canal (hoy son un supuesto de diseño, sección 6) | Requiere trazabilidad de campañas por canal (WhatsApp Business API, tracking de apertura de email/SMS) |


## 9. Supuestos y limitaciones (consolidado)

| # | Supuesto / limitación | Tipo | Sección |
|---|---|---|---|
| 1 | No existe variable objetivo (compra sí/no) — el modelo es no supervisado + reglas, no se puede medir precision/recall real | Limitación estructural del dataset | Todo el notebook |
| 2 | El catálogo de precios de hogar, vehículo y crédito no está confirmado (la web de Colsubsidio no publica esas tarifas) — se documenta el producto pero no se asigna precio | Supuesto por validar | Sección 5 |
| 3 | La propensión a seguros de mascota no se modela de forma proactiva por falta de señal de tenencia de mascota en los datos | Limitación de datos, decisión de diseño | Sección 5 |
| 4 | Los pesos del scoring de canal (app/autogestión/asistido) y de la ventana de tiempo son un diseño razonado, no calibrado con datos reales de conversión por canal | Supuesto por validar | Sección 6 |
| 5 | El proxy "casado con 3 hijos" se mapea a `SEGMENTO_GRUPO_FAMILIAR = FAMILIA NUCLEAR INTEGRAL`, porque la base no tiene el número exacto de hijos | Supuesto por validar | Sección 5 |
| 6 | `CIUDAD_AFILIADO` con 58% de nulos se agrupa en 3 buckets amplios; se pierde granularidad geográfica real | Limitación de datos | Sección 1-2 |
| 7 | K-Prototypes se ajusta sobre una muestra (25k-30k) y se predice sobre el total, en vez de ajustar sobre las 500k/1.5M filas completas — decisión de eficiencia computacional, con impacto marginal esperado dado el tamaño de muestra | Decisión de diseño, a validar con benchmark si se lleva a producción | Sección 4, 7 |
| 8 | Este notebook corre sobre la muestra de 500.000 filas disponible hoy; el pipeline es el mismo para el 1.5M de producción, solo cambia el archivo de entrada | Aclaración de alcance | Todo |


## 10. Métricas de éxito propuestas y próximos pasos

**Métricas de éxito propuestas** *(a validar con el equipo de negocio — no hay ventana de
conversión histórica todavía)*:

- Tasa de apertura/click por canal recomendado, medida por al menos 4-6 semanas antes de
  calibrar los pesos del scoring de canal.
- % de afiliados que adquieren el producto recomendado dentro de una ventana de 30-45 días
  post-contacto (ventana propuesta).
- % de reducción en tiempo de asesor humano por venta cerrada, comparando el flujo autoservido
  vs. el flujo actual asistido.

**Próximos pasos:**

1. Validar con negocio los supuestos (especialmente el catálogo de precios de
   hogar/vehículo y el mapeo de `SEGMENTO_GRUPO_FAMILIAR`).
2. Instrumentar la captura de las columnas de la sección 8, empezando por "seguros ya
   contratados" (mayor impacto, menor esfuerzo de integración).
3. Diseñar el experimento A/B necesario para calibrar los pesos del scoring de canal con datos
   reales de respuesta.
